# Regime-Based Factor Allocation Strategy
## v3 vs v4A Analysis

This notebook demonstrates the regime-based factor allocation strategy with a comparison between:
- **v3**: Baseline regime strategy with full risk-off
- **v4A**: Modified strategy with softened risk-off (max 30%) during Recovery regimes

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('/home/user/claude')

from src.data.data_loader import DataLoader
from src.factors.factor_signals import FactorSignals
from src.regime.regime_detector import RegimeDetector, smooth_regimes
from src.backtest.strategy import RegimeStrategy, BaselineStrategies
from src.utils.performance import annual_returns, monthly_returns_table, rolling_sharpe

# Plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load all data
loader = DataLoader()
data = loader.load_all_data()

prices_d = data['prices_d']
prices_m = data['prices_m']
rets_m = data['rets_m']
fred_m = data['fred_m']

print(f"Data loaded: {len(prices_m)} months ({prices_m.index[0].date()} to {prices_m.index[-1].date()})")
print(f"Tickers: {list(prices_m.columns)}")

## 2. Calculate Factor Signals

In [ ]:
# Calculate momentum and valuation signals
factor_calc = FactorSignals(prices_m, rets_m)
momentum, z_mom, valuation, z_val = factor_calc.calculate_all_signals()

print("Latest Z-Momentum Signals:")
display(z_mom.tail())

print("\nLatest Z-Valuation Signals:")
display(z_val.tail())

## 3. Detect Regimes

In [ ]:
# Detect economic regimes
detector = RegimeDetector(fred_m)
regimes_raw = detector.detect_regime_rule_based()
regimes = smooth_regimes(regimes_raw, min_duration=3)

print("Regime Distribution:")
display(regimes.value_counts())

print("\nLatest Regimes:")
display(regimes.tail(12))

## 4. Run Strategies

In [ ]:
# Baseline strategies
baselines = BaselineStrategies(rets_m, prices_m)

rets_eq, cum_eq, stats_eq = baselines.equal_weight()
rets_mom, cum_mom, stats_mom = baselines.momentum_top2()
rets_spy, cum_spy, stats_spy = baselines.buy_and_hold_spy()

print("Baseline strategies calculated")

In [ ]:
# Regime strategies
strategy = RegimeStrategy(rets_m, prices_m, z_mom, z_val, regimes, fred_m)

rets_v3, cum_v3, stats_v3, w_v3 = strategy.backtest_v3()
rets_v4A, cum_v4A, stats_v4A, w_v4A = strategy.backtest_v4A()

print("Regime strategies calculated")

## 5. Performance Comparison

In [ ]:
# Create performance summary
stats_df = pd.DataFrame([stats_eq, stats_mom, stats_v3, stats_v4A, stats_spy])

display(stats_df.style.format({
    "CAGR": "{:.2%}",
    "Vol": "{:.2%}",
    "Sharpe": "{:.2f}",
    "MDD": "{:.2%}",
    "Calmar": "{:.2f}",
    "Sortino": "{:.2f}",
    "Win Rate": "{:.2%}",
}))

## 6. Visualizations

In [ ]:
# Plot cumulative returns
plt.figure(figsize=(14, 7))
cum_eq.plot(label="Equal-Weight Factors", linewidth=2)
cum_mom.plot(label="Factor Momentum (Top2)", linewidth=2)
cum_v3.plot(label="Regime+Val+Mom (v3)", linewidth=2.5)
cum_v4A.plot(label="Regime+Val+Mom (v4A)", linewidth=2.5, linestyle='--')
cum_spy.plot(label="SPY", linestyle=':', linewidth=2, alpha=0.7)

plt.title("Cumulative Returns Comparison (v3 vs v4A)", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Cumulative Return", fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Annual returns comparison
ann_eq = annual_returns(rets_eq).rename("Equal-Weight")
ann_mom = annual_returns(rets_mom).rename("Momentum Top2")
ann_v3 = annual_returns(rets_v3).rename("v3")
ann_v4A = annual_returns(rets_v4A).rename("v4A")
ann_spy = annual_returns(rets_spy).rename("SPY")

ann_compare = pd.concat([ann_eq, ann_mom, ann_v3, ann_v4A, ann_spy], axis=1).fillna(0.0)

plt.figure(figsize=(14, 7))
ann_compare.plot(kind='bar', width=0.8)
plt.title("Annual Returns by Year", fontsize=16, fontweight='bold')
plt.xlabel("Year", fontsize=12)
plt.ylabel("Return", fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.axhline(y=0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(ann_compare.style.format("{:.2%}"))

In [ ]:
# Regime timeline
fig, ax = plt.subplots(figsize=(14, 4))

regime_colors = {
    'Expansion': 'green',
    'Peak': 'orange',
    'Contraction': 'red',
    'Recovery': 'blue',
    'Unknown': 'gray'
}

for i, (date, regime) in enumerate(regimes.items()):
    next_date = regimes.index[i+1] if i < len(regimes)-1 else date
    ax.axvspan(date, next_date, color=regime_colors.get(regime, 'gray'), alpha=0.3)

ax.set_title("Economic Regime Timeline", fontsize=16, fontweight='bold')
ax.set_xlabel("Date", fontsize=12)
ax.set_yticks([])

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, alpha=0.3, label=regime)
                  for regime, color in regime_colors.items() if regime != 'Unknown']
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

## 7. Deep Dive: v4A Portfolio Composition

In [ ]:
# Plot portfolio weights over time
fig, ax = plt.subplots(figsize=(14, 7))
w_v4A.plot.area(ax=ax, alpha=0.7)
plt.title("v4A Portfolio Weights Over Time", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Weight", fontsize=12)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cash allocation comparison between v3 and v4A
fig, ax = plt.subplots(figsize=(14, 6))
w_v3['BIL'].plot(ax=ax, label='v3 Cash', linewidth=2)
w_v4A['BIL'].plot(ax=ax, label='v4A Cash', linewidth=2, linestyle='--')

# Highlight Recovery regimes
for i, (date, regime) in enumerate(regimes.items()):
    if regime == 'Recovery':
        next_date = regimes.index[i+1] if i < len(regimes)-1 else date
        ax.axvspan(date, next_date, color='blue', alpha=0.1)

plt.title("Cash Allocation: v3 vs v4A (Recovery periods highlighted)", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Cash Weight", fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Risk Analysis

In [ ]:
# Rolling Sharpe ratios
sharpe_v3 = rolling_sharpe(rets_v3, window=12)
sharpe_v4A = rolling_sharpe(rets_v4A, window=12)
sharpe_spy = rolling_sharpe(rets_spy, window=12)

plt.figure(figsize=(14, 6))
sharpe_v3.plot(label='v3', linewidth=2)
sharpe_v4A.plot(label='v4A', linewidth=2, linestyle='--')
sharpe_spy.plot(label='SPY', linewidth=2, linestyle=':', alpha=0.7)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.title("Rolling 12-Month Sharpe Ratio", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Sharpe Ratio", fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown analysis
from src.utils.performance import rolling_drawdown

dd_v3 = rolling_drawdown(rets_v3)
dd_v4A = rolling_drawdown(rets_v4A)
dd_spy = rolling_drawdown(rets_spy)

plt.figure(figsize=(14, 6))
dd_v3.plot(label='v3', linewidth=2)
dd_v4A.plot(label='v4A', linewidth=2, linestyle='--')
dd_spy.plot(label='SPY', linewidth=2, linestyle=':', alpha=0.7)
plt.title("Drawdown Over Time", fontsize=16, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Drawdown", fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Key Insights

### v4A Modification
The v4A strategy implements a **Recovery regime risk-off softening**:
- During Recovery regimes, cash allocation is capped at 30% (vs potential 50-70% in v3)
- This allows maintaining at least 70% factor exposure during recovery periods
- Other regimes (Expansion, Peak, Contraction) follow the same risk-off rules as v3

### Expected Benefits
1. **Higher Returns in Recovery**: More exposure to factor premiums during recovery
2. **Better Participation**: Captures upside momentum during market recoveries
3. **Improved Risk-Adjusted Returns**: Sharpe ratio may improve if recovery periods are profitable

### Potential Risks
1. **Increased Volatility**: Less cash cushion during uncertain recovery periods
2. **False Recovery Signals**: Losses if recovery regime is misclassified
3. **Drawdown Risk**: Potential for larger drawdowns if recovery turns to contraction